In [15]:

import pandas as pd
import re
import ast
from html import escape
from IPython.display import display, HTML

In [4]:
df = pd.read_csv("plenaire_verslagen_relevant_sections.csv", index_col=0)
df.head()

,title,body,year,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits,text,relevant_text,word_count,relevant_word_count
Unnamed: 0,,,,,,,,,,,,,,
74,Geannoteerde besluitenlijst ministerraad 28 me...,MINISTERRAAD\nKenmerk : 4206864\nBESLUITENLIJS...,2021,yes,[],['kunstmatige intelligentie'],['kunstmatige intelligentie'],0,2,[],NaN,Conclusies van de coördinatiecommissie d.d. 25...,5180.0,109.0
76,Agenda ministerraad 4 juni 2021,MINISTERRAAD\nKenmerk : 3753052\nAGENDA\nVerga...,2021,yes,[],"['algoritmen', 'artificiële intelligentie']","['algoritmen', 'artificiële intelligentie']",0,2,[],NaN,Programma Landelijke Vreemdelingen Voorziening...,1219.0,91.0
77,Geannoteerde besluitenlijst ministerraad 4 jun...,MINISTERRAAD\nKenmerk : 4208548\nBESLUITENLIJS...,2021,yes,[],"['ai', 'algoritmen', 'artificiële intelligenti...","['ai', 'algoritmen', 'artificiële intelligenti...",0,6,[],NaN,"1 juni 2021,\nnr.22 (Minister van BZ)\nDe conc...",6588.0,488.0
134,Geannoteerde besluitenlijst ministerraad 29 ok...,MINISTERRAAD\nKenmerk : 4232087\nBESLUITENLIJS...,2021,yes,[],['bard'],['bard'],0,2,[],NaN,4. EU-implementatie\na. Wijziging van het Alge...,6316.0,205.0
148,Geannoteerde besluitenlijst ministerraad 26 no...,MINISTERRAAD\nKenmerk : 4237648\nBESLUITENLIJS...,2021,yes,[],"['ai', 'artificial intelligence']","['ai', 'artificial intelligence']",0,2,[],NaN,Raad Buitenlandse Zaken (Handel) d.d. 29 novem...,5817.0,328.0


In [17]:
import ast

def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

df['matched_keywords_all'] = df.apply(
    lambda row: to_list(row['company_hits']) + to_list(row['matched_keywords_all']),
    axis=1
)
df['matched_keywords_all'].iloc[0]

['kunstmatige intelligentie']

In [11]:


def inspect_ai_related(df, body='relevant_text',  num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """
    
    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', body}.issubset(df.columns):
        missing = {'title', body} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

  
    

    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        
        ai_val = row['ai_related']
        title = row['title']
        body_text = row[body]  # ✅ 'body' parameter stays intact

         # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body_text, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples
    

In [19]:
# program = "all" to display from all programs, or specify a program like "jinek"
inspect_ai_related(df, body = 'relevant_text', num_samples=5, random_state=42)

Displayed 5 random articles (with row-specific matched keywords highlighted).


,title,body,year,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits,text,relevant_text,word_count,relevant_word_count
Unnamed: 0,,,,,,,,,,,,,,
577,Geannoteerde Agenda voor de inzet van Nederlan...,Geannoteerde Agenda voor de inzet van het Koni...,2024,yes,[],"['ai', 'artificial intelligence', 'kunstmatige...","[ai, artificial intelligence, kunstmatige inte...",0,3,[],NaN,Voor het Koninkrijk benadrukken deze mondiale ...,3157.0,333.0
530,Agenda informele Raad Werkgelegenheid en Socia...,GEANNOTEERDE AGENDA INFORMELE RAAD WERKGELEGEN...,2023,yes,[],"['ai', 'algoritmen', 'algoritmes', 'kunstmatig...","[ai, algoritmen, algoritmes, kunstmatige intel...",0,8,[],NaN,Nederland heeft in Europees verband relatief h...,6898.0,721.0
148,Geannoteerde besluitenlijst ministerraad 26 no...,MINISTERRAAD\nKenmerk : 4237648\nBESLUITENLIJS...,2021,yes,[],"['ai', 'artificial intelligence']","[ai, artificial intelligence]",0,2,[],NaN,Raad Buitenlandse Zaken (Handel) d.d. 29 novem...,5817.0,328.0
860,Besluitenlijst ministerraad 28 mei 2025,MINISTERRAAD\nKenmerk : 3825377\nBESLUITENLIJS...,2025,yes,[],['ai'],[ai],0,2,[],NaN,1.2 Fiche 2: Richtlijn periodieke technische c...,1413.0,154.0
978,Ministerie van BZK/VRO Bestuursraadstukken 7 n...,Docnrl\nMinisterie van Binnenlandse Zaken en\n...,2025,yes,[],['ai'],[ai],0,2,[],NaN,o V v e - r v s e te r r b k i e n n d i v n e...,8978.0,721.0
